<a href="https://colab.research.google.com/github/Roxellana/TechStore-Evaluaci-n/blob/main/Evaluaci%C3%B3n_PruebaTecnica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
# ==========================================
# CONFIGURACIÓN DEL ENTORNO (EJECUTAR PRIMERO)
# ==========================================
import pandas as pd
import sqlite3

# 1) URLs de archivos crudos (Raw) en GitHub
urls = {
    'clientes': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/clientes.csv",
    'ventas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/ventas.csv",
    'trabajadores': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/trabajadores.csv",
    'cargos': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/cargos.csv",
    'areas': "https://raw.githubusercontent.com/vbroosing/prueba-tecnica/refs/heads/main/areas.csv"
}

print("Preparando base de datos 'techstore.db'.. .")

# 2) Crear la conexión y la base de datos
conn = sqlite3.connect('techstore.db')

# 3) Leer cada CSV e insertarlo como tabla en la base de datos
for tabla, url in urls.items():
    try:
        df = pd.read_csv(url, sep=None, engine='python')
        df.to_sql(tabla, conn, index=False, if_exists='replace')
        print(f"Tabla '{tabla}' cargada exitosamente.")
    except Exception as e:
        print(f"Error al cargar la tabla '{tabla}': {e}")

conn.close()
print("¡Entorno listo! La base de datos relacional está disponible para ser consultada.")

Preparando base de datos 'techstore.db'.. .
Tabla 'clientes' cargada exitosamente.
Tabla 'ventas' cargada exitosamente.
Tabla 'trabajadores' cargada exitosamente.
Tabla 'cargos' cargada exitosamente.
Tabla 'areas' cargada exitosamente.
¡Entorno listo! La base de datos relacional está disponible para ser consultada.


In [14]:
# VERIFICACIÓN DE DATOS CARGADOS
import sqlite3
import pandas as pd

conn = sqlite3.connect('techstore.db')
cursor = conn.cursor()

print("="*50)
print("VERIFICACIÓN DE BASE DE DATOS")
print("="*50)

# Tablas
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tablas = cursor.fetchall()
print(f"\n Tablas encontradas: {[t[0] for t in tablas]}")

# Registros por tabla
print("\n Conteo de registros:")
for tabla in ['clientes', 'ventas', 'trabajadores', 'cargos', 'areas']:
    cursor.execute(f"SELECT COUNT(*) FROM {tabla}")
    count = cursor.fetchone()[0]
    print(f"   {tabla}: {count}")

# Muestra de ventas
print("\n Muestra de ventas (primeras 5):")
df_ventas = pd.read_sql_query("SELECT id, id_cliente, total_dte, estado FROM ventas LIMIT 5", conn)
print(df_ventas.to_string(index=False))

# Verificar nulos en clientes
print("\n Nulos en tabla clientes:")
df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn)
print(df_clientes.isnull().sum())

conn.close()
print("\n Verificación completada")

VERIFICACIÓN DE BASE DE DATOS

 Tablas encontradas: ['clientes', 'ventas', 'trabajadores', 'cargos', 'areas']

 Conteo de registros:
   clientes: 252
   ventas: 1200
   trabajadores: 35
   cargos: 10
   areas: 5

 Muestra de ventas (primeras 5):
 id  id_cliente  total_dte       estado
  1         191     650631    Entregado
  2         109     903198    Entregado
  3          76     231213    Entregado
  4         158     752982 Sin entregar
  5         132      36388    Entregado

 Nulos en tabla clientes:
id                  0
rut                 0
nombre              0
segundo_nombre      0
apellido            0
segundo_apellido    0
edad                4
fecha_nacimiento    0
dtype: int64

 Verificación completada


In [13]:
# PARTE 1 - CONSULTA 1: Ventas Totales
print("="*50)
print("1. Ventas Totales (Estado 'Entregado')")
print("="*50)

conn = sqlite3.connect('techstore.db')

query = """
SELECT
    SUM(total_dte) AS ingreso_total_entregado,
    ROUND(SUM(total_dte) * 100.0 / (SELECT SUM(total_dte) FROM ventas), 2) AS porcentaje_del_total
FROM ventas
WHERE estado = 'Entregado'
"""

df = pd.read_sql_query(query, conn)
print(f"\n Ingreso total entregado: ${df['ingreso_total_entregado'].values[0]:,.2f}")
print(f" Porcentaje del total de ventas: {df['porcentaje_del_total'].values[0]}%")

conn.close()

1. Ventas Totales (Estado 'Entregado')

 Ingreso total entregado: $477,191,018.00
 Porcentaje del total de ventas: 81.06%


In [5]:
# PARTE 1 - CONSULTA 2: Top 5 Clientes
print("="*50)
print("2. Top 5 Clientes por Gasto Histórico")
print("="*50)

conn = sqlite3.connect('techstore.db')

query = """
SELECT
    c.id,
    c.rut,
    c.nombre || ' ' || c.apellido AS nombre_completo,
    SUM(v.total_dte) AS total_gastado
FROM clientes c
JOIN ventas v ON c.id = v.id_cliente
WHERE v.estado = 'Entregado'
GROUP BY c.id, c.rut, c.nombre, c.apellido
ORDER BY total_gastado DESC
LIMIT 5
"""

df = pd.read_sql_query(query, conn)
print("\n" + df.to_string(index=False))

conn.close()

2. Top 5 Clientes por Gasto Histórico

 id        rut    nombre_completo  total_gastado
145 14605016-6     Paulina Medina        5604037
 82 21866669-8    Sofía Sepúlveda        5466055
 76 16189034-0     Matías Fuentes        5086756
 81 10212267-4 Natalia Valenzuela        5012359
 73 15871360-8   Constanza Torres        4858617


In [6]:
# PARTE 1 - CONSULTA 3: Porcentaje de Retención
print("="*50)
print("3. Porcentaje de Retención (segunda compra ≤200 días)")
print("="*50)

conn = sqlite3.connect('techstore.db')

query = """
WITH primeras_compras AS (
    SELECT
        id_cliente,
        MIN(dte) AS primera_compra_dte
    FROM ventas
    WHERE estado = 'Entregado'
    GROUP BY id_cliente
),
segundas_compras AS (
    SELECT
        v.id_cliente,
        MIN(v.dte) AS segunda_compra_dte
    FROM ventas v
    JOIN primeras_compras p ON v.id_cliente = p.id_cliente
    WHERE v.estado = 'Entregado'
      AND v.dte > p.primera_compra_dte
      AND (v.dte - p.primera_compra_dte) <= 200
    GROUP BY v.id_cliente
)
SELECT
    ROUND(
        (SELECT COUNT(*) FROM segundas_compras) * 100.0 /
        (SELECT COUNT(*) FROM primeras_compras), 2
    ) AS porcentaje_retencion
"""

df = pd.read_sql_query(query, conn)
print(f"\n📈 Porcentaje de clientes que compraron nuevamente: {df['porcentaje_retencion'].values[0]}%")

conn.close()

3. Porcentaje de Retención (segunda compra ≤200 días)

📈 Porcentaje de clientes que compraron nuevamente: 27.46%


In [7]:
# PARTE 2 - LIMPIEZA: Identificar nulos y duplicados
print("="*50)
print("PARTE 2 - LIMPIEZA DE DATOS")
print("="*50)

conn = sqlite3.connect('techstore.db')

df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn)
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)

print("\n📊 VALORES NULOS EN CLIENTES:")
print(df_clientes.isnull().sum())

print("\n📊 DUPLICADOS:")
print(f"   Por id: {df_clientes.duplicated(subset=['id']).sum()}")
print(f"   Por rut: {df_clientes.duplicated(subset=['rut']).sum()}")

# Mostrar el duplicado si existe
duplicados_rut = df_clientes[df_clientes.duplicated(subset=['rut'], keep=False)]
if len(duplicados_rut) > 0:
    print("\n⚠️ RUT duplicado encontrado:")
    print(duplicados_rut[['id', 'rut', 'nombre', 'apellido']].to_string(index=False))

print("\n   CRITERIO DE LIMPIEZA:")
print("   • Segundo_nombre y segundo_apellido: se permiten nulos (campos opcionales)")
print("   • Edad/fecha_nacimiento: si hubiera nulos, imputar con la mediana")
print("   • RUT duplicado: mantener el registro con más datos o el de menor ID")

conn.close()

PARTE 2 - LIMPIEZA DE DATOS

📊 VALORES NULOS EN CLIENTES:
id                  0
rut                 0
nombre              0
segundo_nombre      0
apellido            0
segundo_apellido    0
edad                4
fecha_nacimiento    0
dtype: int64

📊 DUPLICADOS:
   Por id: 0
   Por rut: 2

⚠️ RUT duplicado encontrado:
 id        rut  nombre apellido
  4 10352896-8 Javiera    Silva
104 12626181-0   Diego     Soto
199 10352896-8 Javiera    Silva
251 12626181-0   Diego     Soto

💡 CRITERIO DE LIMPIEZA:
   • Segundo_nombre y segundo_apellido: se permiten nulos (campos opcionales)
   • Edad/fecha_nacimiento: si hubiera nulos, imputar con la mediana
   • RUT duplicado: mantener el registro con más datos o el de menor ID


In [8]:
# PARTE 2 - MÉTRICAS REQUERIDAS
print("="*50)
print("PARTE 2 - MÉTRICAS")
print("="*50)

conn = sqlite3.connect('techstore.db')

df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)

# Top 10 total_dte más altos
print("\n📊 Top 10 montos más altos (total_dte):")
top10 = df_ventas.nlargest(10, 'total_dte')[['id', 'total_dte', 'estado']]
print(top10.to_string(index=False))

# Top 3 clientes por volumen de compras
print("\n📊 Top 3 clientes por volumen de transacciones:")
top3_volumen = df_ventas[df_ventas['estado'] == 'Entregado'] \
    .groupby('id_cliente').size().reset_index(name='cantidad_compras') \
    .nlargest(3, 'cantidad_compras')
print(top3_volumen.to_string(index=False))

# Top 3 vendedores por monto acumulado
print("\n📊 Top 3 vendedores por monto acumulado:")
top3_vendedores = df_ventas[df_ventas['estado'] == 'Entregado'] \
    .groupby('id_vendedor')['total_dte'].sum().reset_index(name='monto_acumulado') \
    .nlargest(3, 'monto_acumulado')
print(top3_vendedores.to_string(index=False))

conn.close()

PARTE 2 - MÉTRICAS

📊 Top 10 montos más altos (total_dte):
 id  total_dte       estado
321     999825    Entregado
484     998087    Entregado
628     996482    Entregado
662     996396    Entregado
459     996276 Sin entregar
115     995217    Entregado
735     995031    Entregado
443     994428    Entregado
179     994032    Entregado
208     992339    Entregado

📊 Top 3 clientes por volumen de transacciones:
 id_cliente  cantidad_compras
        147                10
         15                 9
         73                 9

📊 Top 3 vendedores por monto acumulado:
 id_vendedor  monto_acumulado
           5         34175834
          13         33497166
           3         31535880


In [15]:
# PARTE 2 - TOP 10 PRODUCTOS
print("="*50)
print("Top 10 Productos Más Vendidos")
print("="*50)

import re

conn = sqlite3.connect('techstore.db')
df_ventas = pd.read_sql_query("SELECT * FROM ventas", conn)

def extraer_producto(desc):
    if ',' in desc:
        return desc.split(',')[0].strip()
    return desc.strip()

def extraer_unidades(desc):
    match = re.search(r'(\d+)\s*unidades?', desc)
    return int(match.group(1)) if match else 1

df_ventas['producto'] = df_ventas['descripcion'].apply(extraer_producto)
df_ventas['unidades'] = df_ventas['descripcion'].apply(extraer_unidades)

top_productos = df_ventas.groupby('producto')['unidades'].sum().reset_index()
top_productos = top_productos.sort_values('unidades', ascending=False).head(10)

print("\n" + top_productos.to_string(index=False))

conn.close()
print("\n  PARTES 1 Y 2 COMPLETADAS")

Top 10 Productos Más Vendidos

                         producto  unidades
              3x Hub USB-C 8 en 1        44
3x Silla de Escritorio Ergonómica        43
3x Escáner de Documentos Portátil        42
  1x Mouse Ergonómico Inalámbrico        40
        2x Disco Duro Externo 2TB        40
          2x Laptop Corporate 14'        39
  2x Mouse Ergonómico Inalámbrico        38
1x Impresora Láser Multifuncional        38
1x Escáner de Documentos Portátil        37
              1x Hub USB-C 8 en 1        37

  PARTES 1 Y 2 COMPLETADAS


In [16]:
# EXPORTAR A GOOGLE SHEETS
print("📤 Exportando datos a Google Sheets...")

!pip install gspread --quiet

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

# Crear Google Sheet
import datetime
nombre_sheet = f'TechStore_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}'
sh = gc.create(nombre_sheet)
print(f"✅ Google Sheet creado: {sh.url}")

# Conectar y preparar datos
conn = sqlite3.connect('techstore.db')

# Primeras 100 ventas
df_ventas_100 = pd.read_sql_query("""
    SELECT v.*, c.nombre, c.apellido, c.rut as rut_cliente
    FROM ventas v
    JOIN clientes c ON v.id_cliente = c.id
    ORDER BY v.dte ASC
    LIMIT 100
""", conn)

# Clientes
df_clientes = pd.read_sql_query("SELECT * FROM clientes", conn)

# Vendedores con cargo y área
df_vendedores = pd.read_sql_query("""
    SELECT t.id, t.nombre, t.apellido, t.rut,
           c.nombre_cargo, a.nombre_area
    FROM trabajadores t
    JOIN cargos c ON t.id_cargo = c.id
    JOIN areas a ON c.id_area = a.id
""", conn)

# Ventas entregadas
df_ventas_entregadas = pd.read_sql_query("""
    SELECT * FROM ventas WHERE estado = 'Entregado'
""", conn)

conn.close()

# Función para subir
def subir_sheet(df, nombre, workbook):
    df_clean = df.copy()
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            df_clean[col] = df_clean[col].astype(str).fillna('')
        else:
            df_clean[col] = df_clean[col].fillna(0)

    ws = workbook.add_worksheet(title=nombre, rows=df_clean.shape[0]+1, cols=df_clean.shape[1])
    data = [df_clean.columns.values.tolist()] + df_clean.values.tolist()
    ws.update(data, value_input_option='USER_ENTERED')
    print(f"   ✅ Hoja '{nombre}': {df_clean.shape[0]} filas")

# Subir
subir_sheet(df_ventas_100, 'Ventas', sh)
subir_sheet(df_clientes, 'Clientes', sh)
subir_sheet(df_vendedores, 'Vendedores', sh)
subir_sheet(df_ventas_entregadas, 'Ventas_Entregadas', sh)

# Eliminar hoja por defecto
try:
    sh.del_worksheet(sh.worksheet("Sheet1"))
except:
    print("   (No había hoja Sheet1 para eliminar)")

print(f"\n🎉 ¡EXPORTACIÓN COMPLETADA!")
print(f"📎 Enlace para trabajar en Google Sheets:")
print(sh.url)

📤 Exportando datos a Google Sheets...
✅ Google Sheet creado: https://docs.google.com/spreadsheets/d/18tr5dcpHkQOHcIlOfqKWbfOG26ja_XNMqdlEmgKFjGI
   ✅ Hoja 'Ventas': 100 filas
   ✅ Hoja 'Clientes': 252 filas
   ✅ Hoja 'Vendedores': 35 filas
   ✅ Hoja 'Ventas_Entregadas': 964 filas
   (No había hoja Sheet1 para eliminar)

🎉 ¡EXPORTACIÓN COMPLETADA!
📎 Enlace para trabajar en Google Sheets:
https://docs.google.com/spreadsheets/d/18tr5dcpHkQOHcIlOfqKWbfOG26ja_XNMqdlEmgKFjGI


**Contexto de la Prueba: E-Commerce "TechStore"**

*Escenario: Eres el nuevo analista de datos de TechStore. El equipo ha notado una fluctuación en los ingresos durante el último trimestre y necesita entender qué está pasando.*

**Instrucciones Iniciales:**

1.   La prueba tiene una duración estimada de 45 mins, pero tendrás 72 horas para enviar tus resultados.
2.	Luego de terminar tus ejercicios envíame el enlace en compartir para tu entorno donde fue desarrollado.
3.	Cualquier archivo externo generado debes subirlo y compartir el enlace en una nueva hoja de google colab, en caso de usar hojas de cálculo de Google debes copiar el enlace en otra hoja de google colab, recuerda titular dicho enlace para saber que iré a mirar.
4.	Dentro de la prueba, ve a Archivo > Guardar una copia en Drive para tener tu propia versión editable de esta prueba.
5.	Ejecuta la primera celda de este cuaderno. Esto generará automáticamente una base de datos SQLite llamada techstore.db en este entorno.
6.	El diccionario de datos disponible en la base de datos es:
*   clientes: id, rut, nombre, segundo_nombre, apellido, segundo_apellido, edad, fecha_nacimiento
*   ventas: id, id_cliente, id_vendedor, dte, total_dte, estado, descripcion
*   trabajadores: id, id_cargo, nombre, apellido, rut
*   cargos: id, nombre_cargo, descripcion, id_area
*   areas: id, nombre_area, descripcion

*disclaimer: Puedes usar IA, sin embargo, revisaré cada linea de código para validar la efectividad y también la totalidad del cumplimiento de las respuestas.*

**Parte 1: Conexión y Extracción (SQL & Python)**

*Conéctate a la base de datos techstore.db usando Python (por ejemplo, con la librería sqlite3 o sqlalchemy). Luego, resuelve las siguientes consultas ejecutando SQL puro dentro de tu código Python:*
1.   **Ventas Totales:** Escribe una consulta para calcular el ingreso total generado por órdenes con estado "Entregado" y el % del total de ventas que representa.
2.   **Top Clientes:** Encuentra los 5 usuarios que han gastado más dinero históricamente, mostrando su id, rut y el total_dte gastado.
3.   **Retención:** Calcula el porcentaje de usuarios que realizaron una segunda compra dentro de los 200 dte siguientes (si alguien compró el dte 11721 y compró nuevamente antes ó justamente en el dte 11921 es considerado).

**Parte 2: Análisis Exploratorio (Python)**

*Trae las tablas necesarias a DataFrames de Pandas y resuelve:*

1.	**Limpieza:** Identifica y muestra los valores nulos o duplicados en la tabla de clientes. Comenta tu criterio.
2.	**Métricas:** Identifica el top 10 de total_dte más altos, el top 3 de clientes por volumen de transacciones (cantidad de compras), y el top 3 de vendedores (id_vendedor) con mayor monto acumulado, solo considera vendedores, si hay otros cargos comerciales que han realizado ventas debes ignorarlos.
3.	**Volumen:** Crea un DataFrame del top 10 de productos mas vendidos ordenado de mayor a menor que muestre el id de la venta, nombre del producto y unidades vendidas (deberás procesar la columna descripcion).


**Parte 3: Lógica de Negocio (Hojas de Cálculo / Excel)**

*Para esta sección, deberás ingeniártelas para extraer/descargar los datos de este entorno (ej. generando archivos CSV desde tus DataFrames o desde la DB) e importarlos a Google Sheets o Excel, puedes utilizar la tecnología que estimes conveniente, lo importante es resolver en Excel.*

1.	**Consolidación**: En una hoja nueva, toma una muestra de las primeras 100 ventas según el dte asociado (a menor folio, el documento es más antiguo). Usa fórmulas de búsqueda (BUSCARV, BUSCARX o INDICE/COINCIDIR) para traer el nombre y apellido del cliente y el nombre_cargo del vendedor asociado a esa venta.
2.	**Tabla Dinámica:** Crea una tabla dinámica que muestre el monto total vendido por cada área de la empresa (nombre_area), filtrando solo el estado "Entregado".
3.	**Segmentación:** En tu hoja de ventas, crea la columna "Categoría de Ticket". Usa funciones lógicas para clasificar la venta como "Ticket Alto" (si el total_dte supera el promedio general) o "Ticket Bajo" (si es igual o inferior).


**Opcional (Puntos extra):**

**Presentación al Negocio (Dashboard + Insights)**

1.	**Visualización**: Conecta tus datos exportados a Power BI, Looker o Tableau. Crea un dashboard de una página con:


*   KPI de Ingresos Totales y Ticket Promedio.
*   Gráfico comparativo de rendimiento por categoría/producto.
*   Filtros interactivos.
2.	**Estrategia**: En un documento breve o en celdas de texto al final de este cuaderno, responde:

*   ¿Cuáles fueron tus hallazgos respecto a la fluctuación de ingresos?
*   Propón dos recomendaciones accionables para marketing/ventas.

**Control de versiones (Github):**

1.	**Creación del repositorio:** Entregar los scripts (SQL/Python) subidos a un repositorio propio de GitHub.
